In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Connected to Snowflake!")

In [ ]:
df = session.table("BIKE_ML_DATA")

df.show()

In [ ]:
df = session.table("BIKE_ML_DATA")
df.show()

In [ ]:
print("DATABASE:")
session.sql("SELECT CURRENT_DATABASE()").show()

print("SCHEMA:")
session.sql("SELECT CURRENT_SCHEMA()").show()

print("ROLE:")
session.sql("SELECT CURRENT_ROLE()").show()

print("WAREHOUSE:")
session.sql("SELECT CURRENT_WAREHOUSE()").show()

In [ ]:
session.sql("""
SHOW TABLES IN DATABASE BIKE_SHARING_DB
""").show()

In [ ]:
session.sql("""
SELECT *
FROM BIKE_SHARING_DB.ML_SCHEMA.BIKE_ML_DATA
LIMIT 10
""").show()

In [ ]:
df = session.table("BIKE_SHARING_DB.ML_SCHEMA.BIKE_ML_DATA")
df.show()

In [ ]:
pdf = df.to_pandas()

print("Dataset shape:", pdf.shape)

pdf.head()

In [ ]:
features = [
    "SEASON",
    "YR",
    "MNTH",
    "HOLIDAY",
    "WEEKDAY",
    "WORKINGDAY",
    "WEATHERSIT",
    "TEMP",
    "ATEMP",
    "HUM",
    "WINDSPEED"
]

target = "CNT"

X = pdf[features]
y = pdf[target]

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
y_pred = model.predict(X_test)

print("First 10 predictions:")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 4))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=10
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
rf_pred = rf_model.predict(X_test)

print(rf_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_mae = mean_absolute_error(y_test, rf_pred)

rf_mse = mean_squared_error(y_test, rf_pred)

rf_rmse = rf_mse ** 0.5

rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("MAE :", round(rf_mae, 2))
print("RMSE:", round(rf_rmse, 2))
print("R²  :", round(rf_r2, 4))

In [ ]:
comparison = {
    "Linear Regression": {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    },
    "Random Forest": {
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R2": rf_r2
    }
}

comparison

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="BIKE_SHARING_DB",
    schema_name="ML_SCHEMA"
)

print("Model Registry connected!")

In [ ]:
import snowflake.ml

print(snowflake.ml.__version__)

In [ ]:
!pip show snowflake-ml-python

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="BIKE_SHARING_DB",
    schema_name="ML_SCHEMA"
)

print("Model Registry connected successfully!")

In [ ]:
import importlib.metadata as md

print("snowflake-ml-python version:")
print(md.version("snowflake-ml-python"))

print("\nSnowflake ML import:")
import snowflake.ml
print(snowflake.ml)

In [ ]:
print("Current database:")
print(session.get_current_database())

print("Current schema:")
print(session.get_current_schema())

print("Current role:")
print(session.get_current_role())

In [ ]:
import importlib.metadata as md

print(md.version("snowflake-ml-python"))

In [ ]:
import importlib.metadata as md

version = md.version("snowflake-ml-python")
print("snowflake-ml-python:", version)

In [ ]:
from snowflake.ml.registry import Registry
print("Registry import: OK")

In [ ]:
# Test the trained Random Forest model

print("Model:", rf_model)
print("Number of trees:", rf_model.n_estimators)

# Generate predictions
rf_pred = rf_model.predict(X_test)

print("First 10 actual values:")
print(y_test.head(10).values)

print("\nFirst 10 predicted values:")
print(rf_pred[:10])

In [ ]:
# Create prediction results table

results = X_test.copy()
results["ACTUAL_CNT"] = y_test.values
results["PREDICTED_CNT"] = rf_pred

results["ERROR"] = (
    results["ACTUAL_CNT"] - results["PREDICTED_CNT"]
)

results.head(10)

In [ ]:
prediction_df = results.copy()

session.write_pandas(
    prediction_df,
    table_name="BIKE_RF_PREDICTIONS",
    database="BIKE_SHARING_DB",
    schema="ML_SCHEMA",
    auto_create_table=True,
    overwrite=True
)

print("Prediction table created successfully!")

In [ ]:
prediction_table = session.table(
    "BIKE_SHARING_DB.ML_SCHEMA.BIKE_RF_PREDICTIONS"
)

prediction_table.show(10)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse],
    "R2": [r2, rf_r2]
})

comparison

In [ ]:
print("MODEL COMPARISON")
print("================")
print(comparison.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    rf_pred,
    alpha=0.7
)

# Perfect prediction line
min_value = min(y_test.min(), rf_pred.min())
max_value = max(y_test.max(), rf_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Bike Rentals")
plt.ylabel("Predicted Bike Rentals")
plt.title("Random Forest: Actual vs Predicted")

plt.show()

In [ ]:
errors = y_test - rf_pred

plt.figure(figsize=(8, 5))

plt.hist(errors, bins=30)

plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Random Forest Prediction Error Distribution")

plt.show()

In [ ]:
evaluation_df = pd.DataFrame({
    "MODEL": ["Linear Regression", "Random Forest"],
    "MAE": [mae, rf_mae],
    "RMSE": [rmse, rf_rmse],
    "R2": [r2, rf_r2]
})

session.write_pandas(
    evaluation_df,
    table_name="MODEL_EVALUATION",
    database="BIKE_SHARING_DB",
    schema="ML_SCHEMA",
    auto_create_table=True,
    overwrite=True
)

print("MODEL_EVALUATION table created!")

In [ ]:
session.table(
    "BIKE_SHARING_DB.ML_SCHEMA.MODEL_EVALUATION"
).show()